# 02 — DataFrame Operations

## Learning objectives

In this notebook, we'll practise the core operations used to transform Spark DataFrames:

- Select columns with `select()`.
- Filter rows with `filter()`.
- Add or replace columns with `withColumn()`.
- Build conditional columns with `when()` and `otherwise()`.
- Remove columns with `drop()`.
- Combine operations through method chaining.

Each operation returns a new DataFrame; it does not modify the original DataFrame in place.


## 1. Setup

Create a SparkSession and a small DataFrame so this notebook can be run independently of Notebook 01.

If a SparkSession already exists in this notebook kernel, `getOrCreate()` reuses it.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("DataFrameOperations")
    .getOrCreate()
)

data = [
    (1, "John", 25, "New York"),
    (2, "Alice", 30, "London"),
    (3, "Bob", 22, "Tashkent"),
    (4, "Sarah", 28, "Berlin")
]

columns = ["id", "name", "age", "city"]

df = spark.createDataFrame(data, columns)

df.show()


## 2. Select columns with `select()`

Use `select()` when you need only certain columns from a DataFrame.

The original `df` remains unchanged.


In [ ]:
df_selected = df.select("name", "age", "city")

df_selected.show()


## 3. Filter rows with `filter()`

Use `filter()` to keep rows that satisfy a condition.

For multiple conditions, use `&` for AND and `|` for OR. Put each condition inside parentheses.


In [ ]:
df_filtered = df.filter(col("age") > 25)

df_filtered.show()


In [ ]:
df.filter(
    (col("age") > 25) & (col("city") != "London")
).show()


## 4. Add or replace a column with `withColumn()`

`withColumn()` returns a DataFrame with a new column. If a column with that name already exists, it replaces that column.

Here, we calculate an age five years in the future.


In [ ]:
df_with_future_age = df.withColumn(
    "future_age",
    col("age") + 5
)

df_with_future_age.show()


## 5. Conditional columns with `when()` and `otherwise()`

These functions provide conditional logic similar to SQL's `CASE WHEN`.

- `when(condition, value)` returns the value when the condition is true.
- `.otherwise(value)` supplies the default when the condition is not met.


In [ ]:
df_with_category = df.withColumn(
    "age_category",
    when(col("age") >= 25, "Adult")
    .otherwise("Young")
)

df_with_category.show()


### Multiple conditions

You can chain multiple `when()` calls. Spark checks them in order and uses the first matching condition.


In [ ]:
df_with_age_group = df.withColumn(
    "age_group",
    when(col("age") < 18, "Minor")
    .when(col("age") < 25, "Young Adult")
    .when(col("age") < 40, "Adult")
    .otherwise("Senior")
)

df_with_age_group.show()


## 6. Remove columns with `drop()`

Use `drop()` to return a DataFrame without the specified column.

Remember to assign the result if you want to keep using the DataFrame without that column.


In [ ]:
df_without_city = df.drop("city")

df_without_city.show()


## 7. Separate variables versus method chaining

Each DataFrame operation returns a new DataFrame. You can save each intermediate result in a separate variable:

`df_selected` → `df_filtered` → `df_with_next_age`.

This makes each step explicit and is useful while learning or debugging.


In [ ]:
df_selected = df.select("name", "age", "city")

df_filtered = df_selected.filter(col("age") > 24)

df_with_next_age = df_filtered.withColumn(
    "age_next_year",
    col("age") + 1
)

df_final = df_with_next_age.drop("city")

df_final.show()


### The same logic with method chaining

Method chaining passes the DataFrame returned by one operation directly into the next operation.

The backslash (`\\`) is Python line-continuation syntax. The dot (`.`) calls a method on the DataFrame returned by the previous step.


In [ ]:
df.select("name", "age", "city") \
    .filter(col("age") > 24) \
    .withColumn("age_next_year", col("age") + 1) \
    .drop("city") \
    .show()


## 8. Practice

Using the original `df`, write a chain that:

1. Selects `name`, `age`, and `city`.
2. Keeps people aged 25 or older.
3. Adds an `age_category` column: `"Adult"` for age 25 or older, otherwise `"Young"`.
4. Displays the result.

Try it yourself before looking back at the examples.


In [ ]:
# Your solution:



## Key takeaways

- `select()` chooses columns.
- `filter()` keeps rows that match a condition.
- `withColumn()` adds or replaces a column.
- `when()` and `otherwise()` implement conditional logic.
- `drop()` removes columns.
- Every transformation returns a new DataFrame.
- Separate variables and method chaining can express the same sequence of operations.

**Next:** Notebook 03 covers data cleaning, including null values, duplicates, and data type conversion.
